### pair-specs dataset

In [ ]:
import itertools, random

roles = [
    "A person", "A scientist", "A professor", "An engineer", "A doctor", "A teacher",
    "A student", "A writer", "A photographer", "A chef", "An artist", "A musician",
    "A librarian", "A researcher", "A traveler", "A programmer", "A designer",
    "A journalist", "A nurse", "An architect", "A barista", "A botanist", "A historian",
    "A mathematician", "A physicist", "A biologist", "A chemist", "A geologist",
    "A pilot", "A firefighter", "A dancer", "A singer", "A filmmaker", "A coach",
    "A swimmer", "A hiker", "A gamer", "A carpenter", "A mechanic", "A gardener",
    "A baker", "A tailor", "A calligrapher", "A poet", "A translator", "A curator",
    "A consultant", "A lawyer", "An entrepreneur", "A marketer"
]

activities = [
    "reading a book", "using a laptop", "writing notes", "drinking coffee", "giving a lecture",
    "analyzing data", "studying", "sketching", "taking photos", "cooking", "painting",
    "playing the guitar", "browsing shelves", "conducting an experiment", "packing a bag",
    "debugging code", "drawing a diagram", "interviewing someone", "taking a patient history",
    "reviewing blueprints", "making espresso", "examining a plant", "presenting a talk",
    "solving equations", "testing a hypothesis", "looking through a microscope",
    "mixing chemicals", "reading a map", "checking instruments", "holding a fire hose",
    "practicing choreography", "singing on stage", "editing a film", "coaching a team",
    "swimming laps", "tying hiking boots", "playing a strategy game", "measuring wood",
    "fixing an engine", "watering plants", "kneading dough", "sewing fabric",
    "writing calligraphy", "reciting a poem", "translating a paragraph", "arranging exhibits",
    "advising a client", "arguing a case", "pitching an idea", "planning a campaign"
]

locations = [
    "in a library", "in a classroom", "in a laboratory", "in a cafe", "at home",
    "in an office", "in a coworking space", "in a studio", "on a stage", "in a kitchen",
    "in a museum", "in a park", "on a train", "at the airport", "by the seaside",
    "on a balcony", "in a workshop", "in a greenhouse", "in a conference room",
    "in a lecture hall", "in a bookstore", "on a rooftop", "in a courtyard",
    "in a quiet study room", "in a bustling market"
]

random.seed(42)  # deterministic order for reproducibility

pairs = []
for r, a, l in itertools.product(roles, activities, locations):
    base = f"{r} {a} {l}"
    with_specs = f"{r} wearing spectacles {a} {l}"
    pairs.append((base, with_specs))
    if len(pairs) == 1000:
        break

prompt_pairs = pairs
len(prompt_pairs)

In [ ]:
# pip install -U datasets huggingface_hub
from datasets import Dataset, DatasetDict, Features, Value
from huggingface_hub import login, create_repo, HfApi
import os

features = Features({
    "pair_index": Value("int32"),
    "base_prompt": Value("string"),
    "with_spectacles_prompt": Value("string"),
})

data = {
    "pair_index": list(range(len(prompt_pairs))),
    "base_prompt": [a for a, _ in prompt_pairs],
    "with_spectacles_prompt": [b for _, b in prompt_pairs],
}

ds = Dataset.from_dict(data, features=features).shuffle(seed=42)

# 3) ---- Split train/test (e.g., 80/20) ----
splits = ds.train_test_split(test_size=0.2, seed=42)
dset = DatasetDict({
    "train": splits["train"],
    "test":  splits["test"],
})

# 4) ---- Push to the Hub ----
# Fill these in:
HF_USERNAME = "nirmalendu01"
REPO_NAME   = "spectacles-bias-prompts"  # change if you like
REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"

# Create the repo if it doesn't exist yet (set private=True if you want)
try:
    create_repo(REPO_ID, repo_type="dataset", private=False, exist_ok=True)
except Exception as e:
    print("Note:", e)

# 5) ---- (Optional) Add a dataset card ----
readme = f"""\
# {REPO_NAME}

A minimal prompt-pair dataset for spectacles bias studies in text-to-image diffusion.

## Contents
- `base_prompt`: the neutral prompt
- `with_spectacles_prompt`: the same prompt with "wearing spectacles"
- `pair_index`: stable pairing id

## Splits
- train: {len(dset['train'])} examples
- test:  {len(dset['test'])} examples

## Intended use
- Measure latent deltas (with − without) on UNet outputs
- Learn/text-side debias directions; cross-attention analyses

## License
- Specify your license here (e.g., CC-BY-4.0)
"""
open("README.md", "w", encoding="utf-8").write(readme)

# Upload the README (optional but nice)
api = HfApi()
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card",
)

# 6) ---- Push the DatasetDict ----
dset.push_to_hub(REPO_ID, private=False, commit_message="Initial upload: train/test splits")

print(f"Done. Dataset pushed to: https://huggingface.co/datasets/{REPO_ID}")


### combine with bigger dataset - total =10k

In [ ]:
# pip install -U datasets huggingface_hub
from datasets import load_dataset, Dataset, DatasetDict, Features, Value
from huggingface_hub import login, create_repo, HfApi
import os, random, itertools, json

# ----------------------------
# CONFIG — CHANGE THESE
# ----------------------------
HF_USERNAME = "nirmalendu01"
REPO_NAME   = "text2image-10k-with-spectacles-pairs"
REPO_ID     = f"{HF_USERNAME}/{REPO_NAME}"
PRIVATE     = False

SEED = 42
random.seed(SEED)

N_BASE  = 9000     # rows from jackyhate/text-to-image-2M (go into text column)
N_TOTAL = 10000    # final dataset size
N_SPECS = N_TOTAL - N_BASE  # 1000 rows will come from your pairs (base/with specs as separate rows)

# `prompt_pairs` must exist in scope:
# prompt_pairs = [(base_prompt, with_spectacles_prompt), ...]  # ~100 pairs

# ----------------------------
# 0) AUTH (choose ONE)
# ----------------------------
# Option A: preset token: os.environ["HF_TOKEN"] = "hf_***"
# Option B: interactive:
# login()  # uncomment if you want an interactive prompt

# ----------------------------
# 1) Load 9k prompts from the public dataset (streaming)
# ----------------------------
streaming_ds = load_dataset("jackyhate/text-to-image-2M", split="train", streaming=True)

def extract_text(example):
    """
    Robustly extract a text field:
    - If there's a 'json' string, try to parse and use its 'text' key if present
    - Else use the first string field we find
    """
    if "json" in example and isinstance(example["json"], dict):
        # try:
        #     j = json.loads(example["json"])
            # common keys to try for prompt text
        j=ex["json"]
        for k in ("text", "prompt", "caption", "content"):
            if k in j and isinstance(j[k], str):
                return j[k]
        # except Exception:
        #     pass
    # fallback: first string field
    for k, v in example.items():
        if isinstance(v, str):
            return v
    raise KeyError("Could not find a text field in example.")

base_prompts = []
for ex in streaming_ds:
    try:
        txt = extract_text(ex)
        if txt and txt.strip():
            base_prompts.append(txt.strip())
    except Exception:
        continue
    if len(base_prompts) >= N_BASE:
        break

if len(base_prompts) < N_BASE:
    raise RuntimeError(f"Only collected {len(base_prompts)} base prompts; increase streaming range or check extractor.")

# Optional: light shuffle for variety (spacing will be preserved later)
random.shuffle(base_prompts)

# ----------------------------
# 2) Flatten your pairs into N_SPECS individual rows
#    (both 'base' and 'with spectacles' become separate rows in `text`)
# ----------------------------
if "prompt_pairs" not in globals():
    raise RuntimeError("You must define `prompt_pairs` (list of (base, with_spectacles)).")

if len(prompt_pairs) == 0:
    raise ValueError("`prompt_pairs` is empty.")

# Flatten: [base_1, with_1, base_2, with_2, ...]
flat_pairs = list(itertools.chain.from_iterable(prompt_pairs))  # length = 2 * len(pairs)

# Upsample (cycle) to exactly N_SPECS rows
spec_rows = list(itertools.islice(itertools.cycle(flat_pairs), N_SPECS))

# Optional: shuffle the order of spectacles rows so they’re not paired back-to-back
random.shuffle(spec_rows)

# ----------------------------
# 3) Interleave equally: place one spectacles row every `gap` positions
# ----------------------------
gap = N_TOTAL // N_SPECS  # e.g., 10000 // 1000 = 10
combined = [None] * N_TOTAL

spec_i = 0
for pos in range(0, N_TOTAL, gap):
    if spec_i >= N_SPECS:
        break
    combined[pos] = {
        "text": spec_rows[spec_i],
        "is_from_pair": True,
        "has_spectacles_phrase": ("wearing spectacles" in spec_rows[spec_i].lower()),
        "source": "user_specs",
    }
    spec_i += 1

# Fill remaining slots with base prompts
base_iter = iter(base_prompts)
for i in range(N_TOTAL):
    if combined[i] is None:
        try:
            p = next(base_iter)
        except StopIteration:
            # Shouldn't happen, but pad if needed
            p = random.choice(base_prompts)
        combined[i] = {
            "text": p,
            "is_from_pair": False,
            "has_spectacles_phrase": False,
            "source": "jackyhate/text-to-image-2M",
        }

# Sanity checks
assert len(combined) == N_TOTAL
assert sum(1 for r in combined if r["is_from_pair"]) == N_SPECS

# ----------------------------
# 4) Create DatasetDict with 80/20 split
#    IMPORTANT: shuffle=False to keep equal spacing intact in each split
# ----------------------------
features = Features({
    "text": Value("string"),
    "is_from_pair": Value("bool"),
    "has_spectacles_phrase": Value("bool"),
    "source": Value("string"),
})

ds_all = Dataset.from_list(combined, features=features)

# Keep order; do not shuffle before split if you want spacing preserved.
splits = ds_all.train_test_split(test_size=0.2, seed=SEED, shuffle=False)
dset = DatasetDict({"train": splits["train"], "test": splits["test"]})

print(dset)
print("Counts:",
      "train", len(dset["train"]),
      "test", len(dset["test"]),
      "spectacles train", sum(dset["train"]["is_from_pair"]),
      "spectacles test",  sum(dset["test"]["is_from_pair"]))

# ----------------------------
# 5) Push to the Hub
# ----------------------------
# Create repo if needed
create_repo(REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)

readme = f"""\
# {REPO_NAME}

A 10k **text-only** dataset combining:
- **9,000** prompts sampled (streaming) from `jackyhate/text-to-image-2M`
- **1,000** rows drawn from user-provided *spectacles* **pairs** (both base and with "wearing spectacles" versions are included as separate rows)

## Schema
- `text`: the prompt
- `is_from_pair`: whether this row came from the spectacles pairs
- `has_spectacles_phrase`: whether the text explicitly includes "wearing spectacles"
- `source`: `"jackyhate/text-to-image-2M"` or `"user_specs"`

## Construction
- Spectacles rows are **equally spaced** in the 10k corpus (one every ~{gap} rows).
- 80/20 split performed **without shuffling** to preserve spacing.

## Intended Use
- Bias/steering analyses for T2I diffusion, cross-attention & UNet activation studies.

## License
- Base data inherits license of `jackyhate/text-to-image-2M`.
- User spectacles prompts: specify your license here.
"""
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

api = HfApi()
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card",
)

# Push datasets
dset.push_to_hub(REPO_ID, private=PRIVATE, commit_message="10k text dataset with equally spaced spectacles rows")
print(f"Done. Dataset pushed to: https://huggingface.co/datasets/{REPO_ID}")